# Simulador base

Reencena os pedidos históricos da Olist (2016–2018) como se acontecessem a partir de 2026-09-06 e grava um lote diário por entidade em `olist.landing.raw_data/<entidade>/sim_date=AAAA-MM-DD/`.

**Parâmetros**

| Widget | Padrão | Uso |
|---|---|---|
| `dias_por_execucao` | `1` | quantos dias simulados esta execução gera (1 a 30) |
| `sim_date_override` | vazio | reprocessa um dia específico (`AAAA-MM-DD`); vazio = continua de onde parou |
| `run_id` | vazio | identificador da execução; no job recebe `{{job.run_id}}`, vazio gera um UUID |

O progresso fica em `olist.ops.simulator_runs`. Desenho: `docs/design/fase-1-simulador-bronze.md`.

In [0]:
# O QUE FAZ: cria os 3 parâmetros do notebook. Eles aparecem como caixas de texto no topo da página.
#   Na execução interativa, você preenche à mão; no job, os valores vêm dos parâmetros da task.
#
# FUNÇÕES:
# - dbutils.widgets.text(nome, valor_padrão, rótulo): cria um widget de texto. "nome" é usado no
#   código para ler o valor; "rótulo" é o texto exibido na tela. Se o widget já existe, nada muda.
#   O valor é lido depois com dbutils.widgets.get(nome) e sempre chega como string.
dbutils.widgets.text("dias_por_execucao", "1", "Dias por execução")
dbutils.widgets.text("sim_date_override", "", "Reprocessar dia (AAAA-MM-DD)")
dbutils.widgets.text("run_id", "", "ID da execução")

In [0]:
# O QUE FAZ: define as constantes do simulador: caminhos dos Volumes, tabela de controle e regras
#   de datas. Todos os caminhos derivam de CATALOG: trocar o catálogo é mudar uma linha só.
#
# RECURSOS:
# - datetime.date(ano, mês, dia): data de calendário sem horário (tipo nativo do Python).
# - f"...{variavel}...": f-string, monta texto substituindo as variáveis entre chaves.
# - /Volumes/<catálogo>/<schema>/<volume>: caminho de arquivos de um Volume do Unity Catalog,
#   a forma de guardar arquivos no serverless.
from datetime import date

CATALOG = "olist"

SOURCE_DIR = f"/Volumes/{CATALOG}/landing/olist_source"
RAW_DIR = f"/Volumes/{CATALOG}/landing/raw_data"
STAGING_DIR = f"{RAW_DIR}/_staging"  # prefixo "_": o Auto Loader ignora a pasta

RUNS_TABLE = f"{CATALOG}.ops.simulator_runs"

DATE_OFFSET_DAYS = 3654          # 522 semanas: preserva o dia da semana
SIM_START = date(2026, 9, 6)     # primeiro pedido (2016-09-04) + 3654 dias
MAX_DIAS_POR_EXECUCAO = 30

In [0]:
# O QUE FAZ: lê os 3 widgets, converte para os tipos certos (int, date, str) e valida.
#   Um valor inválido para a execução aqui, antes de qualquer leitura ou escrita.
#
# FUNÇÃO DECLARADA:
# - parse_params(dias_raw, override_raw, run_id_raw): recebe os textos dos widgets e devolve
#   (dias, override, run_id). É pura: não usa Spark nem dbutils, então pode ser testada com pytest.
#   Regras: dias inteiro de 1 a 30; override vazio ou data >= SIM_START; run_id vazio vira UUID.
#
# FUNÇÕES NATIVAS:
# - uuid.uuid4(): gera um identificador aleatório único (ex.: "3f2b...-..."); str() vira texto.
# - str.strip(): remove espaços e quebras de linha do início e do fim do texto.
# - int(texto): converte texto em inteiro; lança ValueError se não for número.
# - date.fromisoformat("AAAA-MM-DD"): converte texto em data; lança ValueError se o formato estiver errado.
# - raise ... from None: lança o novo erro escondendo o erro original, para a mensagem ficar limpa.
# - {valor!r}: mostra o valor com repr(), com aspas, o que deixa visível um texto vazio ou com espaços.
# - "a or b": devolve "a" se ele tiver valor (texto não vazio); senão, devolve "b".
# - dbutils.widgets.get(nome): lê o valor atual do widget (sempre string).
import uuid


def parse_params(dias_raw: str, override_raw: str, run_id_raw: str) -> tuple[int, date | None, str]:
    """Converte e valida os valores dos widgets. Não usa Spark nem dbutils: pode ser testada isoladamente."""
    try:
        dias = int(dias_raw.strip())
    except ValueError:
        raise ValueError(f"dias_por_execucao deve ser um inteiro; recebido: {dias_raw!r}") from None
    if not 1 <= dias <= MAX_DIAS_POR_EXECUCAO:
        raise ValueError(f"dias_por_execucao deve estar entre 1 e {MAX_DIAS_POR_EXECUCAO}; recebido: {dias}")

    override = None
    if override_raw.strip():
        try:
            override = date.fromisoformat(override_raw.strip())
        except ValueError:
            raise ValueError(f"sim_date_override deve estar no formato AAAA-MM-DD; recebido: {override_raw!r}") from None
        if override < SIM_START:
            raise ValueError(f"sim_date_override não pode ser anterior a {SIM_START}; recebido: {override}")

    run_id = run_id_raw.strip() or str(uuid.uuid4())
    return dias, override, run_id


DIAS_POR_EXECUCAO, SIM_DATE_OVERRIDE, RUN_ID = parse_params(
    dbutils.widgets.get("dias_por_execucao"),
    dbutils.widgets.get("sim_date_override"),
    dbutils.widgets.get("run_id"),
)

print(f"run_id:            {RUN_ID}")
print(f"dias_por_execucao: {DIAS_POR_EXECUCAO}")
print(f"sim_date_override: {SIM_DATE_OVERRIDE or '(nenhum: continua de onde parou)'}")

## Leitura da fonte e deslocamento de datas

Lê os 9 CSVs originais de `olist_source` com todas as colunas como texto e desloca as colunas de data em `DATE_OFFSET_DAYS` dias. Nada é gravado nesta etapa: as transformações do Spark são *lazy* e só executam quando um resultado é pedido.

In [0]:
# O QUE FAZ: mapeia cada entidade da landing para o CSV de origem e lista as colunas de data/hora
#   que precisam ser deslocadas em 3.654 dias. Só constantes: nenhum dado é lido aqui.
#
# RECURSOS:
# - dict {chave: valor}: dicionário; SOURCE_FILES["orders"] devolve o nome do arquivo.
# - TS_FORMAT: padrão de data/hora do Spark ("yyyy" ano, "MM" mês, "dd" dia, "HH" hora 0-23,
#   "mm" minuto, "ss" segundo). Atenção: "MM" é mês e "mm" é minuto.
# Chaves = nomes das entidades na landing (desenho A2)
SOURCE_FILES = {
    "orders": "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "products": "olist_products_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "category_translation": "product_category_name_translation.csv",
}

TS_FORMAT = "yyyy-MM-dd HH:mm:ss"

# Colunas de data/hora por entidade; as demais entidades não têm datas
DATE_COLUMNS = {
    "orders": [
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
    ],
    "order_items": ["shipping_limit_date"],
    "order_reviews": ["review_creation_date", "review_answer_timestamp"],
}

In [0]:
# O QUE FAZ: declara as duas funções de leitura: ler o CSV de uma entidade e deslocar as datas.
#
# FUNÇÕES DECLARADAS:
# - read_source(entity): devolve um DataFrame com o CSV original da entidade, tudo como string.
# - shift_dates(df, columns, days): soma "days" dias às colunas de data indicadas e devolve o
#   resultado de novo como texto. Se a lista de colunas estiver vazia, devolve o df sem mudança.
#
# FUNÇÕES NATIVAS:
# - spark.read.option(chave, valor).csv(caminho): leitor de CSV. Opções usadas:
#     header=True     -> a primeira linha tem os nomes das colunas;
#     multiLine=True  -> permite campo entre aspas com quebra de linha (comentários das avaliações);
#     escape='"'      -> aspas dentro de um campo vêm duplicadas (""), padrão do CSV da Olist.
#   Sem inferSchema, todas as colunas são string.
# - df.withColumns({nome: expressão}): cria ou substitui várias colunas de uma vez (uma projeção
#   só, mais eficiente que encadear vários withColumn).
# - F.col(nome): referência a uma coluna. F.lit(valor): valor constante como coluna.
# - F.to_timestamp_ntz(coluna, formato): texto -> TIMESTAMP_NTZ (data/hora SEM fuso). Sem fuso,
#   o horário de verão e o fuso da sessão não alteram o horário ao somar dias.
# - F.make_dt_interval(days=n): cria um intervalo de n dias; "timestamp + intervalo" soma os dias.
# - F.date_format(timestamp, formato): data/hora -> texto no formato indicado.
# - {c: ... for c in columns}: dict comprehension, monta o dicionário com uma expressão por coluna.
from pyspark.sql import DataFrame
from pyspark.sql import functions as F


def read_source(entity: str) -> DataFrame:
    """Lê o CSV original da entidade. Sem inferSchema: todas as colunas chegam como string."""
    return (
        spark.read
        .option("header", True)
        .option("multiLine", True)  # comentários das avaliações têm quebra de linha
        .option("escape", '"')
        .csv(f"{SOURCE_DIR}/{SOURCE_FILES[entity]}")
    )


def shift_dates(df: DataFrame, columns: list[str], days: int) -> DataFrame:
    """Desloca colunas de data/hora em `days` dias. Entrada e saída em texto no formato TS_FORMAT; nulos continuam nulos."""
    if not columns:
        return df
    return df.withColumns({
        c: F.date_format(
            F.to_timestamp_ntz(F.col(c), F.lit(TS_FORMAT)) + F.make_dt_interval(days=F.lit(days)),
            TS_FORMAT,
        )
        for c in columns
    })

In [0]:
# O QUE FAZ: monta o DataFrame de cada entidade de origem, já com as datas deslocadas.
#   É lazy (preguiçoso): o Spark só registra o plano. Lê apenas a primeira linha de cada CSV para
#   descobrir os nomes das colunas; os dados são lidos quando uma ação (count, write...) roda.
#
# RECURSOS:
# - DATE_COLUMNS.get(entity, []): busca no dicionário; se a entidade não tem datas, devolve [].
# - ", ".join(lista): junta os textos da lista separados por vírgula.
sources = {
    entity: shift_dates(read_source(entity), DATE_COLUMNS.get(entity, []), DATE_OFFSET_DAYS)
    for entity in SOURCE_FILES
}

print(f"{len(sources)} entidades carregadas: {', '.join(sources)}")

In [0]:
# O QUE FAZ: valida o deslocamento de datas (critério de aceite nº 5): todo pedido deve ter
#   andado exatamente 3.654 dias, no mesmo dia da semana, de 2026-09-06 a 2028-10-18.
#
# FUNÇÕES NATIVAS:
# - df.select(...): escolhe ou calcula colunas. .alias(nome): renomeia a coluna resultante.
# - df.transform(funcao, *args): chama funcao(df, *args); permite encadear funções próprias.
# - F.to_date(coluna): texto ou timestamp -> DATE (descarta o horário).
# - df.agg(...): agrega a tabela inteira em uma linha.
# - F.datediff(fim, inicio): diferença em dias entre duas datas.
# - F.min / F.max: menor e maior valor.
# - F.count_if(condição): conta as linhas em que a condição é verdadeira.
# - F.dayofweek(data): dia da semana (1 = domingo ... 7 = sábado).
# - display(df): ação do Databricks; executa o plano e mostra o resultado em tabela.
# - df.limit(n): só as n primeiras linhas.
validacao = (
    read_source("orders")
    .select(F.col("order_purchase_timestamp").alias("original"), "order_purchase_timestamp")
    .transform(shift_dates, ["order_purchase_timestamp"], DATE_OFFSET_DAYS)
    .select(
        F.to_date("original").alias("data_original"),
        F.to_date("order_purchase_timestamp").alias("data_simulada"),
    )
    .agg(
        F.min(F.datediff("data_simulada", "data_original")).alias("min_dias"),
        F.max(F.datediff("data_simulada", "data_original")).alias("max_dias"),
        F.count_if(F.dayofweek("data_simulada") != F.dayofweek("data_original")).alias("dia_semana_diferente"),
        F.min("data_simulada").alias("primeira_compra"),
        F.max("data_simulada").alias("ultima_compra"),
    )
)

display(validacao)
display(sources["orders"].limit(5))

## Montagem das entidades

Transforma as 9 tabelas de origem nas 10 entidades da landing (desenho A2). Cada entidade recebe a coluna `sim_date`, o dia simulado em que o registro é emitido:

| Entidade | `sim_date` |
|---|---|
| `orders` (cabeçalho) | dia da compra |
| `order_status_events` | dia de cada evento (`created`, `approved`, `shipped`, `delivered` e status final) |
| `order_items`, `order_payments` | dia da compra do pedido |
| `order_reviews` | dia da resposta da avaliação |
| `customers` (CDC) | `INSERT` no dia do primeiro pedido do cliente; `UPDATE` e `DELETE` acrescentados na seção "Mudanças cadastrais" |
| `sellers` (CDC) | `INSERT` na carga inicial (`SIM_START`); `UPDATE` e `DELETE` acrescentados na seção "Mudanças cadastrais" |
| `products`, `geolocation`, `category_translation` | carga inicial no primeiro dia (`SIM_START`) |

Registros com `sim_date` posterior ao último dia do dataset nunca são emitidos, porque a execução só gera dias até esse limite (Parte 4).

In [0]:
# O QUE FAZ: constantes da montagem das entidades: início da simulação como texto de data/hora,
#   status finais que viram evento extra e as colunas do cabeçalho do pedido (desenho A2).
#
# RECURSOS:
# - list [...]: lista ordenada. Inteiros como 5 são constantes nomeadas para não espalhar
#   "números mágicos" pelo código.
SIM_START_TS = f"{SIM_START} 00:00:00"

# Status finais que não têm coluna de data própria: viram um evento extra (sequência 5)
FINAL_STATUSES = ["canceled", "unavailable", "invoiced", "processing"]
FINAL_EVENT_SEQUENCE = 5

ORDER_HEADER_COLUMNS = [
    "order_id",
    "customer_id",
    "order_purchase_timestamp",
    "order_estimated_delivery_date",
]

In [0]:
# O QUE FAZ: declara as funções puras (DataFrame -> DataFrame) que transformam as 9 tabelas de
#   origem nas 10 entidades da landing, cada uma com a coluna sim_date (dia em que é emitida).
#
# FUNÇÕES DECLARADAS:
# - with_sim_date(df, ts_column): cria sim_date com a data (sem horário) da coluna indicada.
# - with_initial_load(df): sim_date = SIM_START para todas as linhas (carga inicial no 1º dia).
# - with_cdc_insert(df, change_ts): acrescenta as colunas de CDC do registro de inclusão:
#   operation = "INSERT", change_ts = momento da mudança, change_sequence = 1.
# - build_orders(orders): cabeçalho do pedido, só as colunas conhecidas na compra; dia = compra.
# - build_status_events(orders): transforma as datas do pedido em eventos (uma linha por status):
#   created(1), approved(2), shipped(3), delivered(4); data nula = etapa que não aconteceu, sem
#   evento. Pedidos com status final (canceled etc.) ganham um evento 5 na última data conhecida.
# - build_order_children(df, orders): itens/pagamentos recebem o sim_date da compra do pedido.
# - build_customers(customers, orders): cliente emitido como CDC INSERT no dia do seu pedido.
# - build_entities(sources): aplica as funções acima e devolve o dicionário das 10 entidades.
#
# FUNÇÕES NATIVAS:
# - Column: tipo de uma expressão de coluna do Spark (usado só na anotação de tipo).
# - df.withColumn(nome, expressão): cria ou substitui uma coluna.
# - F.stack(n, v1, v2, ...): gera n linhas a partir de cada linha de entrada, distribuindo os
#   valores em grupos (aqui 4 grupos de 3 valores: status, sequência, data). É um "unpivot".
#   .alias("a", "b", "c") dá nome às 3 colunas geradas.
# - df.where(condição): filtra linhas. Column.isNotNull(): verdadeiro se não for nulo.
# - Column.isin(lista): verdadeiro se o valor estiver na lista.
# - F.greatest(c1, c2, ...): o maior valor entre as colunas, ignorando nulos (nulo só se todas
#   forem nulas). Funciona em texto porque "yyyy-MM-dd HH:mm:ss" ordena como data.
# - df1.unionByName(df2): empilha as linhas casando as colunas pelo NOME (union casa pela posição).
# - df.join(outro, on=coluna, how="inner"): junta as tabelas pela coluna; inner mantém só as
#   linhas com correspondência nos dois lados.
# - df.drop(coluna): remove a coluna.
from pyspark.sql import Column


def with_sim_date(df: DataFrame, ts_column: str) -> DataFrame:
    """Define o dia simulado de emissão a partir de uma coluna de data/hora em texto."""
    return df.withColumn("sim_date", F.to_date(ts_column))


def with_initial_load(df: DataFrame) -> DataFrame:
    """Carga inicial: todos os registros emitidos no primeiro dia simulado."""
    return df.withColumn("sim_date", F.lit(SIM_START))


def with_cdc_insert(df: DataFrame, change_ts: Column) -> DataFrame:
    """Colunas de CDC para o registro de inclusão (desenho A2)."""
    return df.withColumns({
        "operation": F.lit("INSERT"),
        "change_ts": change_ts,
        "change_sequence": F.lit(1),
    })


def build_orders(orders: DataFrame) -> DataFrame:
    """Cabeçalho do pedido: só o que se conhece no momento da compra."""
    return with_sim_date(orders.select(*ORDER_HEADER_COLUMNS), "order_purchase_timestamp")


def build_status_events(orders: DataFrame) -> DataFrame:
    """Um registro por mudança de status. Datas nulas (etapa que não aconteceu) não geram evento."""
    timeline = (
        orders
        .select(
            "order_id",
            F.stack(
                F.lit(4),
                F.lit("created"),   F.lit(1), F.col("order_purchase_timestamp"),
                F.lit("approved"),  F.lit(2), F.col("order_approved_at"),
                F.lit("shipped"),   F.lit(3), F.col("order_delivered_carrier_date"),
                F.lit("delivered"), F.lit(4), F.col("order_delivered_customer_date"),
            ).alias("status", "event_sequence", "event_ts"),
        )
        .where(F.col("event_ts").isNotNull())
    )

    final_status = (
        orders
        .where(F.col("order_status").isin(FINAL_STATUSES))
        .select(
            "order_id",
            F.col("order_status").alias("status"),
            F.lit(FINAL_EVENT_SEQUENCE).alias("event_sequence"),
            # última data conhecida do pedido; greatest ignora nulos
            F.greatest(
                "order_purchase_timestamp",
                "order_approved_at",
                "order_delivered_carrier_date",
                "order_delivered_customer_date",
            ).alias("event_ts"),
        )
    )

    return with_sim_date(timeline.unionByName(final_status), "event_ts")


def build_order_children(df: DataFrame, orders: DataFrame) -> DataFrame:
    """Itens e pagamentos: emitidos no dia da compra do pedido a que pertencem."""
    purchase_dates = build_orders(orders).select("order_id", "sim_date")
    return df.join(purchase_dates, on="order_id", how="inner")


def build_customers(customers: DataFrame, orders: DataFrame) -> DataFrame:
    """Cliente nasce no checkout: inclusão (CDC INSERT) no dia do pedido."""
    first_order = orders.select("customer_id", "order_purchase_timestamp")
    return (
        customers
        .join(first_order, on="customer_id", how="inner")
        .transform(with_cdc_insert, F.col("order_purchase_timestamp"))
        .transform(with_sim_date, "order_purchase_timestamp")
        .drop("order_purchase_timestamp")
    )


def build_entities(sources: dict[str, DataFrame]) -> dict[str, DataFrame]:
    """Monta as 10 entidades da landing a partir das 9 tabelas de origem (datas já deslocadas)."""
    orders = sources["orders"]
    return {
        "orders": build_orders(orders),
        "order_status_events": build_status_events(orders),
        "order_items": build_order_children(sources["order_items"], orders),
        "order_payments": build_order_children(sources["order_payments"], orders),
        "order_reviews": with_sim_date(sources["order_reviews"], "review_answer_timestamp"),
        "customers": build_customers(sources["customers"], orders),
        "sellers": with_initial_load(with_cdc_insert(sources["sellers"], F.lit(SIM_START_TS))),
        "products": with_initial_load(sources["products"]),
        "geolocation": with_initial_load(sources["geolocation"]),
        "category_translation": with_initial_load(sources["category_translation"]),
    }

In [0]:
# O QUE FAZ: monta as 10 entidades. Continua lazy: nenhum dado é processado ainda.
entities = build_entities(sources)

print(f"{len(entities)} entidades montadas: {', '.join(entities)}")

In [0]:
# O QUE FAZ: confere se as contagens de linhas das entidades batem com as do setup (Fase 0).
#   order_status_events não tem valor esperado aqui (é derivada) e aparece com ok = true.
#
# FUNÇÕES NATIVAS:
# - df.count(): ação; executa o plano e devolve o número de linhas.
# - spark.createDataFrame(lista_de_tuplas, schema): cria um DataFrame pequeno a partir de dados
#   Python. O schema é uma string DDL: "nome tipo, nome tipo".
# - EXPECTED_ROWS.get(name): valor esperado, ou None se a entidade não estiver no dicionário.
# - Column.isNull(): verdadeiro se for nulo.
# - "|" entre colunas: OU lógico do Spark (cada lado entre parênteses, por precedência de operadores).
# - 99_441: o "_" é só separador visual de milhar em Python.
# Mesmas contagens validadas no setup; order_status_events é derivada e tem contagem própria
EXPECTED_ROWS = {
    "orders": 99_441,
    "order_items": 112_650,
    "order_payments": 103_886,
    "order_reviews": 99_224,
    "customers": 99_441,
    "sellers": 3_095,
    "products": 32_951,
    "geolocation": 1_000_163,
    "category_translation": 71,
}

contagens = spark.createDataFrame(
    [(name, df.count(), EXPECTED_ROWS.get(name)) for name, df in entities.items()],
    "entidade string, linhas long, esperado long",
).withColumn("ok", F.col("esperado").isNull() | (F.col("linhas") == F.col("esperado")))

display(contagens)

In [0]:
# O QUE FAZ: valida os eventos de status: quantidade por sequência/status e, para um pedido
#   entregue, os 4 eventos em dias diferentes (critério de aceite nº 6).
#
# FUNÇÕES NATIVAS:
# - df.groupBy(colunas).count(): agrupa e conta as linhas de cada grupo (coluna "count").
# - df.orderBy(colunas): ordena o resultado.
display(
    entities["order_status_events"]
    .groupBy("event_sequence", "status")
    .count()
    .orderBy("event_sequence", "status")
)

# Pedido entregue: 4 eventos, cada um no seu dia (critério de aceite nº 6)
display(
    entities["order_status_events"]
    .where(F.col("order_id") == "e481f51cbdc54678b7cc49136f2d6af7")
    .orderBy("event_sequence")
)

## Injeção de falhas (caos)

Configuração e ferramentas das falhas da origem (desenho A5). Esta seção **não altera nenhum dado**: declara a configuração, as funções de sorteio e a gravação do gabarito, usadas pelas seções seguintes.

- **Sorteio por registro:** hash da chave do registro (`xxhash64`) combinado com a semente e com o tipo de falha. O mesmo registro recebe sempre a mesma decisão, em qualquer execução e em qualquer ordem de processamento.
- **Sorteio por arquivo:** `random.Random` com semente em texto (semente, tipo de falha, entidade e dia).
- **Gabarito:** cada falha injetada vira uma linha em `olist.ops.simulator_chaos_log`. Ao reprocessar um dia, as linhas desse dia são substituídas (`replaceWhere`), como acontece com os arquivos da landing.
- `CHAOS_CONFIG["enabled"] = False` desliga todas as falhas sem alterar o código.

In [0]:
# O QUE FAZ: concentra em um só lugar todas as taxas, datas e entidades de cada falha (desenho A5).
#   As chaves do dicionário são os mesmos nomes de chaos_type gravados no gabarito.
#   CHAOS_RECORD_KEYS define a chave que identifica cada registro: entra no hash do sorteio e na
#   coluna record_key do gabarito.
#
# RECURSOS:
# - frozenset({...}): conjunto que não pode ser alterado depois de criado.
# - dict aninhado: CHAOS_CONFIG["late_arrival"]["pct"] lê a taxa de atraso.
# - date(ano, mês, dia): datas fixas das falhas de schema e de valor.
CHAOS_LOG_TABLE = f"{CATALOG}.ops.simulator_chaos_log"

# Entidades com arquivo diário (as cargas estáticas não recebem reenvio nem duplicata: desenho A5)
DAILY_ENTITIES = frozenset({
    "orders",
    "order_status_events",
    "order_items",
    "order_payments",
    "order_reviews",
    "customers",
    "sellers",
})

CHAOS_CONFIG = {
    "enabled": True,
    "seed": "olist-lakehouse",  # trocar a semente muda todas as escolhas
    "file_resend": {"pct": 5, "entities": DAILY_ENTITIES},
    "duplicate_record": {"pct": 1, "entities": DAILY_ENTITIES},
    "late_arrival": {"pct": 3, "max_days": 3, "entities": frozenset({"order_status_events", "order_items"})},
    "out_of_order": {"entities": frozenset({"order_status_events"})},
    "new_column": {"entity": "order_payments", "column": "payment_gateway", "from_date": date(2026, 10, 8)},
    "bad_value": {"entity": "order_items", "column": "price", "date": date(2026, 10, 7), "pct": 10},
    "corrupt_file": {"entity": "order_items", "date": date(2026, 10, 10)},
}

# Chave de cada registro: usada no sorteio e na coluna record_key do gabarito
CHAOS_RECORD_KEYS = {
    "orders": ["order_id"],
    "order_status_events": ["order_id", "event_sequence"],
    "order_items": ["order_id", "order_item_id"],
    "order_payments": ["order_id", "payment_sequential"],
    "order_reviews": ["review_id", "order_id"],
    "customers": ["customer_id", "change_sequence"],
    "sellers": ["seller_id", "change_sequence"],
}

In [0]:
# O QUE FAZ: declara as funções que decidem, sem aleatoriedade real, quais registros e arquivos
#   recebem cada falha. Mesma entrada -> mesma decisão, em qualquer execução.
#
# FUNÇÕES DECLARADAS:
# - chaos_hash(keys, salt): expressão Spark com o hash da semente, do "sal" e das colunas-chave.
#   O sal é o nome da falha: sem ele, os 1% duplicados seriam sempre os mesmos registros dos 3%
#   atrasados, e as falhas deixariam de ser independentes.
# - record_selected(keys, salt, pct): coluna booleana, verdadeira para cerca de pct% dos registros.
# - record_days(keys, salt, max_days): coluna inteira de 1 a max_days, fixa por registro (dias de
#   atraso). Usa um sal próprio, para que o número de dias não dependa da escolha do registro.
# - file_selected(entity, day, salt, pct): True para cerca de pct% dos arquivos (entidade + dia).
#
# FUNÇÕES NATIVAS:
# - F.xxhash64(c1, c2, ...): hash de 64 bits das colunas; algoritmo fixo, dá o mesmo resultado em
#   qualquer cluster, sessão ou no SQL. O hash() do Python NÃO serve: para textos, ele muda a cada
#   processo (PYTHONHASHSEED), e a escolha mudaria a cada execução.
# - F.pmod(a, n): resto da divisão sempre positivo (0 a n-1); o % do Spark pode ser negativo.
# - random.Random(semente_texto): gerador com semente própria; com semente em texto, a sequência é
#   sempre a mesma. .random() devolve um número entre 0 e 1.
# - [F.col(k) for k in keys]: lista de colunas; o * a "desempacota" como argumentos da função.
# - timedelta: importado aqui porque a validação logo abaixo monta uma lista de dias (o import da
#   seção de controle vem depois; importar duas vezes não tem efeito).
import random
from datetime import timedelta


def chaos_hash(keys: list[str], salt: str) -> Column:
    """Hash determinístico das colunas-chave, combinado com a semente e o tipo de falha."""
    return F.xxhash64(F.lit(CHAOS_CONFIG["seed"]), F.lit(salt), *[F.col(k) for k in keys])


def record_selected(keys: list[str], salt: str, pct: int) -> Column:
    """Verdadeiro para cerca de pct% dos registros; sempre o mesmo resultado para a mesma chave."""
    if not CHAOS_CONFIG["enabled"]:
        return F.lit(False)
    return F.pmod(chaos_hash(keys, salt), F.lit(100)) < pct


def record_days(keys: list[str], salt: str, max_days: int) -> Column:
    """De 1 a max_days, fixo para cada chave."""
    return F.pmod(chaos_hash(keys, f"{salt}-days"), F.lit(max_days)) + 1


def file_selected(entity: str, day: date, salt: str, pct: int) -> bool:
    """True para cerca de pct% dos arquivos; sempre o mesmo resultado para a mesma entidade e dia."""
    if not CHAOS_CONFIG["enabled"]:
        return False
    return random.Random(f"{CHAOS_CONFIG['seed']}-{salt}-{entity}-{day}").random() * 100 < pct

In [0]:
# O QUE FAZ: declara as funções que montam e gravam o gabarito das falhas injetadas.
#
# FUNÇÕES DECLARADAS:
# - chaos_log_from_records(df, chaos_type, entity, detail): uma linha de gabarito por registro
#   afetado, com a chave do registro em texto ("order_id|order_item_id"). Função pura.
# - chaos_log_from_files(rows): gabarito das falhas por arquivo (reenvio, arquivo corrompido),
#   a partir de uma lista Python; record_key fica nulo.
# - write_chaos_log(log, start, end): grava o gabarito dos dias [start, end], substituindo o que
#   existia nesses dias. Devolve o número de linhas gravadas.
#
# FUNÇÕES NATIVAS:
# - F.concat_ws("|", c1, c2): junta as colunas em um texto, separadas por "|".
# - Column.cast("string"): converte para texto.
# - spark.createDataFrame(lista, schema_ddl): DataFrame pequeno a partir de dados Python; funciona
#   com lista vazia porque o schema é informado.
# - df.write.mode("overwrite").option("replaceWhere", condição).saveAsTable(tabela): substitui
#   SOMENTE as linhas que atendem à condição, em uma única transação (commit Delta). Rodar duas
#   vezes dá o mesmo resultado: é a forma idempotente de reprocessar um intervalo. As demais linhas,
#   o schema e os comentários da tabela são mantidos.
# - F.current_timestamp(): horário real da gravação (logged_at).
CHAOS_LOG_COLUMNS = ["sim_date", "chaos_type", "entity", "record_key", "detail"]
CHAOS_LOG_SCHEMA = "sim_date date, chaos_type string, entity string, record_key string, detail string"


def chaos_log_from_records(df: DataFrame, chaos_type: str, entity: str, detail: Column) -> DataFrame:
    """Gabarito das falhas por registro: uma linha por registro afetado."""
    return df.select(
        "sim_date",
        F.lit(chaos_type).alias("chaos_type"),
        F.lit(entity).alias("entity"),
        F.concat_ws("|", *[F.col(k).cast("string") for k in CHAOS_RECORD_KEYS[entity]]).alias("record_key"),
        detail.cast("string").alias("detail"),
    )


def chaos_log_from_files(rows: list[tuple[date, str, str, str]]) -> DataFrame:
    """Gabarito das falhas por arquivo. Cada item: (sim_date, chaos_type, entity, detail)."""
    return spark.createDataFrame(
        [(sim_date, chaos_type, entity, None, detail) for sim_date, chaos_type, entity, detail in rows],
        CHAOS_LOG_SCHEMA,
    )


def write_chaos_log(log: DataFrame, start: date, end: date) -> int:
    """Substitui o gabarito dos dias [start, end] pelo conteúdo de `log`. Devolve as linhas gravadas."""
    window_filter = f"sim_date BETWEEN '{start}' AND '{end}'"
    (
        log
        .where(window_filter)
        .withColumns({"run_id": F.lit(RUN_ID), "logged_at": F.current_timestamp()})
        .select("run_id", *CHAOS_LOG_COLUMNS, "logged_at")
        .write
        .mode("overwrite")
        .option("replaceWhere", window_filter)
        .saveAsTable(CHAOS_LOG_TABLE)
    )
    return spark.table(CHAOS_LOG_TABLE).where(window_filter).count()

In [0]:
# O QUE FAZ: confere o sorteio em order_items, sem gravar nada: taxas próximas de 1% e 3%,
#   independência entre as falhas (poucos registros nas duas), dias de atraso de 1 a 3 e a
#   decisão de quatro registros conhecidos, que deve ser idêntica à calculada fora do notebook.
#   Mostra também os arquivos sorteados para reenvio no período de 2026-10-05 a 2026-10-12.
#
# FUNÇÕES NATIVAS:
# - F.count_if(condição): conta as linhas em que a condição é verdadeira; "&" é o E lógico.
# - F.min / F.max: menor e maior valor; ignoram nulos (F.when sem otherwise devolve nulo).
# - F.when(condição, valor): valor quando a condição é verdadeira, nulo nos demais casos.
# - date + timedelta(days=i): o i-ésimo dia a partir da data.
items_keys = CHAOS_RECORD_KEYS["order_items"]
sorteio = entities["order_items"].select(
    *items_keys,
    record_selected(items_keys, "duplicate_record", CHAOS_CONFIG["duplicate_record"]["pct"]).alias("duplicado"),
    record_selected(items_keys, "late_arrival", CHAOS_CONFIG["late_arrival"]["pct"]).alias("atrasado"),
    record_days(items_keys, "late_arrival", CHAOS_CONFIG["late_arrival"]["max_days"]).alias("dias_atraso"),
)

display(sorteio.agg(
    F.count("*").alias("itens"),
    F.count_if("duplicado").alias("duplicados"),
    F.count_if("atrasado").alias("atrasados"),
    F.count_if(F.col("duplicado") & F.col("atrasado")).alias("nas_duas_falhas"),
    F.min(F.when(F.col("atrasado"), F.col("dias_atraso"))).alias("min_dias"),
    F.max(F.when(F.col("atrasado"), F.col("dias_atraso"))).alias("max_dias"),
))

display(
    sorteio
    .where(F.col("order_item_id") == "1")
    .where(F.col("order_id").isin(
        "0036757472ece3dde52fd4bfd929c90e",
        "00042b26cf59d7ce69dfabb4e55b4fd9",
        "0009c9a17f916a706d71784483a5d643",
        "06661c686ea79767c8ab8b8ffa3e62fd",
    ))
    .orderBy("order_id")
)

for i in range(8):
    day = date(2026, 10, 5) + timedelta(days=i)
    sorteados = sorted(e for e in DAILY_ENTITIES if file_selected(e, day, "file_resend", CHAOS_CONFIG["file_resend"]["pct"]))
    print(f"{day}: {', '.join(sorteados) or '-'}")

In [0]:
# O QUE FAZ: mostra como ficam as linhas do gabarito, sem gravar nada: os itens de 2026-10-07
#   sorteados para o valor fora do padrão (vírgula decimal em price) e uma falha por arquivo.
#
# FUNÇÕES NATIVAS:
# - df.unionByName(outro): empilha as linhas casando as colunas pelo nome.
# - F.concat(...): junta textos; F.col("price") entra no detalhe como valor original.
bad = CHAOS_CONFIG["bad_value"]
exemplo = chaos_log_from_records(
    entities["order_items"]
    .where(F.col("sim_date") == bad["date"])
    .where(record_selected(items_keys, "bad_value", bad["pct"])),
    "bad_value",
    "order_items",
    F.concat(F.lit("price original: "), F.col("price")),
).unionByName(
    chaos_log_from_files([(date(2026, 10, 9), "file_resend", "order_items", "order_items_2026-10-09_part-000_retry-1.json")])
)

display(exemplo.orderBy("chaos_type", "record_key"))

## Mudanças cadastrais (CDC)

Acrescenta aos feeds de `customers` e `sellers` as mudanças sintéticas definidas no desenho (A6). As mudanças são **dados legítimos** da origem, e não falhas: não dependem de `CHAOS_CONFIG["enabled"]` e não vão para o gabarito. Usam o mesmo hash determinístico da seção anterior.

| Feed | Operação | Abrangência | Momento |
|---|---|---|---|
| `customers` | `UPDATE` (endereço) | 3% | inclusão + 7 a 120 dias |
| `customers` | `DELETE` (LGPD) | 0,5% | última data conhecida do pedido + 30 a 365 dias |
| `sellers` | `UPDATE` (endereço) | 10% | qualquer dia após a carga inicial |
| `sellers` | `DELETE` (desligamento) | 2% | última venda + 30 a 90 dias |

- O `UPDATE` traz o estado completo, com um endereço (CEP, cidade e UF) escolhido entre os endereços existentes da própria entidade.
- O `DELETE` traz só a chave; os demais atributos ficam nulos.
- `change_sequence`: 1 (`INSERT`), 2 (`UPDATE`), 3 (`DELETE`). Nenhuma mudança depois da exclusão, nenhuma depois do último dia da simulação.

In [0]:
# O QUE FAZ: define as taxas e os prazos das mudanças, calcula o último dia da simulação (SIM_END,
#   antes calculado no controle da execução, agora necessário aqui) e declara as funções auxiliares.
#   Taxas em "por mil" porque 0,5% não cabe em um sorteio de 0 a 99.
#
# FUNÇÕES DECLARADAS:
# - hash_selected(keys, salt, per_mille): coluna booleana, verdadeira para cerca de per_mille/1000
#   das chaves. Diferente de record_selected, não obedece a CHAOS_CONFIG["enabled"]: CDC não é falha.
# - hash_days(keys, salt, min_days, max_days): coluna inteira de min_days a max_days, fixa por chave.
# - add_days(ts, days): soma dias (coluna) a uma data/hora em texto e devolve texto no TS_FORMAT,
#   mesma técnica de shift_dates (TIMESTAMP_NTZ, sem fuso).
# - address_catalog(df, address_cols): endereços distintos da entidade, numerados de 0 a n-1 em
#   ordem fixa, para que o sorteio do endereço novo seja reproduzível.
# - with_new_address(df, keys, salt, catalog, address_cols): troca o endereço de cada linha por um
#   endereço do catálogo escolhido pelo hash da chave.
#
# FUNÇÕES NATIVAS:
# - F.pmod(hash, 1000): número de 0 a 999. Column.cast("int"): make_dt_interval exige inteiro de
#   32 bits, e o hash é de 64 bits.
# - Window.orderBy(colunas) + F.row_number().over(janela): numera as linhas em ordem (1, 2, 3...).
#   Sem partitionBy, o Spark junta tudo em uma partição: aceitável aqui (cerca de 15 mil linhas).
# - df.distinct(): remove linhas repetidas.
# - df.count(): ação; tamanho do catálogo, usado como divisor do sorteio.
# - df.drop(*colunas): remove as colunas; o * desempacota a lista.
# - (fim - início).days: diferença em dias entre duas datas do Python.
from pyspark.sql import Window

CDC_CONFIG = {
    "customers": {
        "update": {"per_mille": 30, "min_days": 7, "max_days": 120},
        "delete": {"per_mille": 5, "min_days": 30, "max_days": 365},
    },
    "sellers": {
        "update": {"per_mille": 100},  # qualquer dia após a carga inicial (até SIM_END)
        "delete": {"per_mille": 20, "min_days": 30, "max_days": 90},
    },
}

# Último dia da simulação: dia da última compra (nenhuma mudança é emitida depois dele)
SIM_END = entities["orders"].agg(F.max("sim_date").alias("fim")).first()["fim"]


def hash_selected(keys: list[str], salt: str, per_mille: int) -> Column:
    """Verdadeiro para cerca de per_mille/1000 das chaves; sempre o mesmo resultado para a mesma chave."""
    return F.pmod(chaos_hash(keys, salt), F.lit(1000)) < per_mille


def hash_days(keys: list[str], salt: str, min_days: int, max_days: int) -> Column:
    """De min_days a max_days, fixo para cada chave."""
    return (F.pmod(chaos_hash(keys, f"{salt}-days"), F.lit(max_days - min_days + 1)) + min_days).cast("int")


def add_days(ts: Column, days: Column) -> Column:
    """Soma dias a uma data/hora em texto (TS_FORMAT). Nulo continua nulo."""
    return F.date_format(F.to_timestamp_ntz(ts, F.lit(TS_FORMAT)) + F.make_dt_interval(days=days), TS_FORMAT)


def address_catalog(df: DataFrame, address_cols: list[str]) -> DataFrame:
    """Endereços distintos, numerados de 0 a n-1 em ordem fixa."""
    return (
        df.select(*address_cols)
        .distinct()
        .withColumn("address_index", F.row_number().over(Window.orderBy(*address_cols)) - 1)
    )


def with_new_address(df: DataFrame, keys: list[str], salt: str, catalog: DataFrame, address_cols: list[str]) -> DataFrame:
    """Substitui o endereço de cada linha por um endereço do catálogo escolhido pelo hash da chave."""
    size = catalog.count()
    return (
        df.drop(*address_cols)
        .withColumn("address_index", F.pmod(chaos_hash(keys, f"{salt}-address"), F.lit(size)))
        .join(catalog, on="address_index", how="inner")
        .drop("address_index")
    )

In [0]:
# O QUE FAZ: declara as funções puras que geram as linhas UPDATE e DELETE de cada feed.
#
# FUNÇÕES DECLARADAS:
# - build_changes(base, keys, address_cols, catalog, salt): parte comum aos dois feeds. Recebe as
#   linhas de INSERT com duas colunas extras, update_ts e delete_ts (nulas quando não há mudança),
#   e devolve as linhas UPDATE (endereço novo, sequência 2) e DELETE (só a chave, sequência 3).
#   Aplica as regras de ordem: UPDATE só se for anterior ao DELETE; nada depois de SIM_END.
# - build_customer_changes(customers, orders): UPDATE na inclusão + 7 a 120 dias; DELETE na última
#   data conhecida do pedido + 30 a 365 dias.
# - build_seller_changes(sellers, order_items, orders): UPDATE em qualquer dia após a carga
#   inicial; DELETE na última venda + 30 a 90 dias. Vendedor sem venda não é desligado.
#
# FUNÇÕES NATIVAS:
# - F.when(condição, valor): valor quando a condição é verdadeira; nulo nos demais casos.
# - "texto" < "texto": datas no formato yyyy-MM-dd HH:mm:ss se comparam corretamente como texto.
# - df1.unionByName(df2, allowMissingColumns=True): empilha casando pelo nome; colunas que faltam
#   em um dos lados (os atributos do DELETE) são preenchidas com nulo.
# - F.greatest(...): a maior data entre as colunas, ignorando nulos.
# - df.groupBy(col).agg(F.max(...)): maior valor por grupo (última venda de cada vendedor).
# - df.join(..., how="left"): mantém todas as linhas da esquerda, mesmo sem correspondência.
def build_changes(base: DataFrame, keys: list[str], address_cols: list[str], catalog: DataFrame, salt: str) -> DataFrame:
    """Linhas UPDATE e DELETE a partir das datas sorteadas (update_ts, delete_ts)."""
    feed_columns = [c for c in base.columns if c not in ("update_ts", "delete_ts")]

    updates = (
        base
        .where(F.col("update_ts").isNotNull())
        .where(F.col("delete_ts").isNull() | (F.col("update_ts") < F.col("delete_ts")))
        .transform(with_new_address, keys, salt, catalog, address_cols)
        .withColumns({"operation": F.lit("UPDATE"), "change_ts": F.col("update_ts"), "change_sequence": F.lit(2)})
    )
    deletes = (
        base
        .where(F.col("delete_ts").isNotNull())
        .select(*keys, F.lit("DELETE").alias("operation"), F.col("delete_ts").alias("change_ts"), F.lit(3).alias("change_sequence"))
    )
    return (
        updates.select(*[c for c in feed_columns if c != "sim_date"])
        .unionByName(deletes, allowMissingColumns=True)
        .transform(with_sim_date, "change_ts")
        .where(F.col("sim_date") <= SIM_END)
        .select(*feed_columns)
    )


def build_customer_changes(customers: DataFrame, orders: DataFrame) -> DataFrame:
    """Mudanças de endereço e exclusões (LGPD) de clientes."""
    keys = ["customer_id"]
    address_cols = ["customer_zip_code_prefix", "customer_city", "customer_state"]
    update, delete = CDC_CONFIG["customers"]["update"], CDC_CONFIG["customers"]["delete"]

    last_known = orders.select(
        "customer_id",
        F.greatest(
            "order_purchase_timestamp",
            "order_approved_at",
            "order_delivered_carrier_date",
            "order_delivered_customer_date",
        ).alias("last_known_ts"),
    )
    base = (
        customers
        .join(last_known, on="customer_id", how="inner")
        .withColumns({
            "update_ts": F.when(
                hash_selected(keys, "customers-update", update["per_mille"]),
                add_days(F.col("change_ts"), hash_days(keys, "customers-update", update["min_days"], update["max_days"])),
            ),
            "delete_ts": F.when(
                hash_selected(keys, "customers-delete", delete["per_mille"]),
                add_days(F.col("last_known_ts"), hash_days(keys, "customers-delete", delete["min_days"], delete["max_days"])),
            ),
        })
        .drop("last_known_ts")
    )
    return build_changes(base, keys, address_cols, address_catalog(customers, address_cols), "customers-update")


def build_seller_changes(sellers: DataFrame, order_items: DataFrame, orders: DataFrame) -> DataFrame:
    """Mudanças de endereço e desligamentos de vendedores."""
    keys = ["seller_id"]
    address_cols = ["seller_zip_code_prefix", "seller_city", "seller_state"]
    update, delete = CDC_CONFIG["sellers"]["update"], CDC_CONFIG["sellers"]["delete"]
    max_update_days = (SIM_END - SIM_START).days

    last_sale = (
        order_items
        .join(orders.select("order_id", "order_purchase_timestamp"), on="order_id", how="inner")
        .groupBy("seller_id")
        .agg(F.max("order_purchase_timestamp").alias("last_sale_ts"))
    )
    base = (
        sellers
        .join(last_sale, on="seller_id", how="left")
        .withColumns({
            "update_ts": F.when(
                hash_selected(keys, "sellers-update", update["per_mille"]),
                add_days(F.lit(SIM_START_TS), hash_days(keys, "sellers-update", 1, max_update_days)),
            ),
            "delete_ts": F.when(
                hash_selected(keys, "sellers-delete", delete["per_mille"]),
                add_days(F.col("last_sale_ts"), hash_days(keys, "sellers-delete", delete["min_days"], delete["max_days"])),
            ),
        })
        .drop("last_sale_ts")
    )
    return build_changes(base, keys, address_cols, address_catalog(sellers, address_cols), "sellers-update")

In [0]:
# O QUE FAZ: acrescenta as mudanças aos feeds customers e sellers. Parte sempre só das linhas
#   INSERT: rodar a célula de novo não acumula mudanças repetidas.
#
# RECURSOS:
# - {**entities, "chave": valor}: novo dicionário com tudo de entities, substituindo as duas chaves.
# - Continua lazy: nada é processado até uma ação.
customers_insert = entities["customers"].where(F.col("operation") == "INSERT")
sellers_insert = entities["sellers"].where(F.col("operation") == "INSERT")

entities = {
    **entities,
    "customers": customers_insert.unionByName(build_customer_changes(customers_insert, sources["orders"])),
    "sellers": sellers_insert.unionByName(build_seller_changes(sellers_insert, sources["order_items"], sources["orders"])),
}

print(f"Último dia da simulação (SIM_END): {SIM_END}")

In [0]:
# O QUE FAZ: confere os feeds sem gravar nada: quantidade por operação (total e até 2026-10-31,
#   período gerado na Fase 1) e as regras do desenho A6, cada uma contada como violação (todas
#   devem ser 0):
#   - mudança depois do DELETE da mesma chave;
#   - mudança depois de SIM_END;
#   - DELETE com algum atributo preenchido;
#   - UPDATE com o mesmo endereço do INSERT;
#   - chave repetida com a mesma change_sequence.
#
# FUNÇÕES NATIVAS:
# - F.max(F.when(cond, valor)).over(Window.partitionBy(chave)): maior valor da chave, considerando
#   só as linhas da condição (aqui, o change_ts do DELETE), repetido em todas as linhas da chave.
# - F.coalesce(a, b, ...): o primeiro valor não nulo; é nulo só se todos forem nulos. Aqui detecta
#   um DELETE com qualquer atributo preenchido.
# - df.alias("x") + F.col("x.coluna"): nomeia os lados do join para distinguir colunas iguais.
# - F.count_distinct(c1, c2): quantidade de combinações distintas.
PHASE_1_END = date(2026, 10, 31)  # último dia gerado na Fase 1 (só para a validação)


def validate_cdc(feed: DataFrame, key: str, address_cols: list[str]) -> DataFrame:
    """Contagens por operação e violações das regras do desenho A6 (todas devem ser 0)."""
    attributes = [c for c in feed.columns if c not in (key, "operation", "change_ts", "change_sequence", "sim_date")]
    with_delete = feed.withColumn(
        "delete_ts", F.max(F.when(F.col("operation") == "DELETE", F.col("change_ts"))).over(Window.partitionBy(key))
    )
    inserts = feed.where(F.col("operation") == "INSERT").alias("i")
    updates = feed.where(F.col("operation") == "UPDATE").alias("u")
    same_address = (
        updates.join(inserts, on=key, how="inner")
        .where(F.expr(" AND ".join(f"u.{c} <=> i.{c}" for c in address_cols)))
        .count()
    )
    return (
        with_delete
        .agg(
            F.count_if(F.col("operation") == "UPDATE").alias("updates"),
            F.count_if((F.col("operation") == "UPDATE") & (F.col("sim_date") <= PHASE_1_END)).alias("updates_ate_31_10"),
            F.count_if(F.col("operation") == "DELETE").alias("deletes"),
            F.count_if((F.col("operation") == "DELETE") & (F.col("sim_date") <= PHASE_1_END)).alias("deletes_ate_31_10"),
            F.count_if((F.col("operation") != "DELETE") & (F.col("change_ts") >= F.col("delete_ts"))).alias("mudanca_apos_delete"),
            F.count_if(F.col("sim_date") > SIM_END).alias("apos_sim_end"),
            F.count_if((F.col("operation") == "DELETE") & F.coalesce(*attributes).isNotNull()).alias("delete_com_atributo"),
            (F.count("*") - F.count_distinct(key, "change_sequence")).alias("chave_sequencia_repetida"),
        )
        .withColumn("update_mesmo_endereco", F.lit(same_address))
    )


display(validate_cdc(entities["customers"], "customer_id", ["customer_zip_code_prefix", "customer_city", "customer_state"]))
display(validate_cdc(entities["sellers"], "seller_id", ["seller_zip_code_prefix", "seller_city", "seller_state"]))

# Um vendedor com UPDATE e DELETE: INSERT (1), UPDATE (2) e DELETE só com a chave (3)
exemplo_seller = (
    entities["sellers"]
    .where(F.col("operation") == "DELETE")
    .join(entities["sellers"].where(F.col("operation") == "UPDATE").select("seller_id"), on="seller_id", how="left_semi")
    .select("seller_id")
    .orderBy("seller_id")
    .limit(1)
)
display(entities["sellers"].join(exemplo_seller, on="seller_id", how="left_semi").orderBy("change_sequence"))

## Falhas por registro

Aplica as falhas que alteram registros (desenho A5) e monta o gabarito correspondente. O resultado fica em `landing_entities`, separado de `entities`:

- `entities`: o que **de fato aconteceu** (pedidos, eventos e mudanças cadastrais).
- `landing_entities`: o que **a origem envia**, com as falhas. É o que a escrita grava na landing.

| Falha | Entidades | Efeito |
|---|---|---|
| `late_arrival` | `order_status_events`, `order_items` | 3% dos registros vão para o arquivo de 1 a 3 dias depois; as datas do evento não mudam |
| `bad_value` | `order_items` | em 2026-10-07, 10% dos `price` com vírgula decimal |
| `new_column` | `order_payments` | a partir de 2026-10-08, os arquivos trazem `payment_gateway` |
| `duplicate_record` | entidades diárias | 1% dos registros repetidos no mesmo arquivo |
| `out_of_order` | `order_status_events` | linhas do arquivo ordenadas por hash, e não pela chave (aplicado na escrita) |

A ordem de aplicação importa: o atraso vem primeiro, porque muda o dia do registro, e todas as outras falhas e o gabarito usam o dia em que o registro **chega**.

In [0]:
# O QUE FAZ: declara as duas falhas que valem para várias entidades. Cada função recebe o
#   DataFrame de uma entidade e devolve (DataFrame com a falha, gabarito da falha).
#
# FUNÇÕES DECLARADAS:
# - inject_late_arrival(df, entity): muda o sim_date dos registros sorteados para 1 a 3 dias
#   depois, só se o novo dia não passar de SIM_END. As colunas de data do registro (event_ts,
#   shipping_limit_date) continuam as originais: é isso que caracteriza a chegada atrasada.
#   No gabarito, sim_date é o dia em que o registro chega; o detalhe traz o dia original.
# - inject_duplicates(df, entity): repete os registros sorteados (a cópia fica no mesmo dia, ou
#   seja, no mesmo arquivo).
#
# FUNÇÕES NATIVAS:
# - F.date_add(data, dias): soma dias a uma data; os dias precisam ser inteiros de 32 bits,
#   por isso o .cast("int") no resultado de record_days.
# - F.when(cond, a).otherwise(b): a quando a condição é verdadeira, b nos demais casos.
# - Colunas com "_" no início (_original_date, _is_late): auxiliares, removidas antes do retorno.
# - F.datediff(fim, início): diferença em dias.
# - F.concat(...): junta textos; números e datas entram com .cast("string").
# - df.unionByName(outro): acrescenta as cópias ao DataFrame original.
def inject_late_arrival(df: DataFrame, entity: str) -> tuple[DataFrame, DataFrame]:
    """Atrasa cerca de 3% dos registros em 1 a 3 dias, sem passar de SIM_END."""
    cfg = CHAOS_CONFIG["late_arrival"]
    keys = CHAOS_RECORD_KEYS[entity]
    late_date = F.date_add("sim_date", record_days(keys, "late_arrival", cfg["max_days"]).cast("int"))

    marked = df.withColumns({
        "_original_date": F.col("sim_date"),
        "_is_late": record_selected(keys, "late_arrival", cfg["pct"]) & (late_date <= F.lit(SIM_END)),
    })
    delayed = marked.withColumn("sim_date", F.when(F.col("_is_late"), late_date).otherwise(F.col("sim_date")))

    log = chaos_log_from_records(
        delayed.where("_is_late"),
        "late_arrival",
        entity,
        F.concat(
            F.lit("atraso de "), F.datediff("sim_date", "_original_date").cast("string"),
            F.lit(" dia(s); dia original "), F.col("_original_date").cast("string"),
        ),
    )
    return delayed.drop("_original_date", "_is_late"), log


def inject_duplicates(df: DataFrame, entity: str) -> tuple[DataFrame, DataFrame]:
    """Repete cerca de 1% dos registros no mesmo arquivo."""
    copies = df.where(record_selected(CHAOS_RECORD_KEYS[entity], "duplicate_record", CHAOS_CONFIG["duplicate_record"]["pct"]))
    log = chaos_log_from_records(copies, "duplicate_record", entity, F.lit("registro repetido no mesmo arquivo"))
    return df.unionByName(copies), log

In [0]:
# O QUE FAZ: declara as falhas de data fixa e o gabarito do embaralhamento.
#
# FUNÇÕES DECLARADAS:
# - inject_bad_value(df): no dia configurado, troca o ponto decimal por vírgula no price dos
#   registros sorteados ("59.90" -> "59,90"). O gabarito guarda o valor original.
# - inject_new_column(df): cria payment_gateway, preenchida a partir da data configurada e nula
#   antes dela. A escrita grava os dias anteriores SEM a coluna (célula da escrita), para que a
#   mudança de schema apareça de fato só a partir dessa data.
# - out_of_order_log(df, entity): gabarito do embaralhamento, uma linha por dia com arquivo
#   (record_key nulo). O embaralhamento em si é aplicado na escrita, na ordenação das linhas.
#
# FUNÇÕES NATIVAS:
# - F.regexp_replace(coluna, padrão, substituto): substitui pelo padrão (expressão regular);
#   r"\." é o ponto literal (sem a barra, "." significaria qualquer caractere).
# - F.array(...) + F.element_at(array, posição): escolhe um item da lista; a posição começa em 1.
# - F.lit(None).cast("string"): nulo do tipo texto.
# - df.select("sim_date").distinct(): os dias que têm registros.
PAYMENT_GATEWAYS = ["pagseguro", "mercadopago", "cielo", "stone"]


def inject_bad_value(df: DataFrame) -> tuple[DataFrame, DataFrame]:
    """Vírgula decimal em parte dos valores do dia configurado."""
    cfg = CHAOS_CONFIG["bad_value"]
    keys = CHAOS_RECORD_KEYS[cfg["entity"]]
    marked = df.withColumn(
        "_is_bad",
        (F.col("sim_date") == F.lit(cfg["date"])) & record_selected(keys, "bad_value", cfg["pct"]),
    )
    log = chaos_log_from_records(
        marked.where("_is_bad"),
        "bad_value",
        cfg["entity"],
        F.concat(F.lit(f"{cfg['column']} original: "), F.col(cfg["column"])),
    )
    changed = marked.withColumn(
        cfg["column"],
        F.when(F.col("_is_bad"), F.regexp_replace(cfg["column"], r"\.", ",")).otherwise(F.col(cfg["column"])),
    )
    return changed.drop("_is_bad"), log


def inject_new_column(df: DataFrame) -> tuple[DataFrame, DataFrame]:
    """Coluna nova a partir da data configurada (nula antes; a escrita omite a coluna nesses dias)."""
    cfg = CHAOS_CONFIG["new_column"]
    if not CHAOS_CONFIG["enabled"]:
        return df, chaos_log_from_files([])
    keys = CHAOS_RECORD_KEYS[cfg["entity"]]
    gateway = F.element_at(
        F.array(*[F.lit(g) for g in PAYMENT_GATEWAYS]),
        (F.pmod(chaos_hash(keys, "new_column"), F.lit(len(PAYMENT_GATEWAYS))) + 1).cast("int"),
    )
    changed = df.withColumn(cfg["column"], F.when(F.col("sim_date") >= F.lit(cfg["from_date"]), gateway))
    log = chaos_log_from_files([
        (cfg["from_date"], "new_column", cfg["entity"], f"coluna {cfg['column']} presente a partir deste arquivo"),
    ])
    return changed, log


def out_of_order_log(df: DataFrame, entity: str) -> DataFrame:
    """Gabarito do embaralhamento: uma linha por dia com arquivo."""
    return df.select("sim_date").distinct().select(
        "sim_date",
        F.lit("out_of_order").alias("chaos_type"),
        F.lit(entity).alias("entity"),
        F.lit(None).cast("string").alias("record_key"),
        F.lit("linhas do arquivo ordenadas por hash, e não pela chave").alias("detail"),
    )

In [0]:
# O QUE FAZ: aplica as falhas na ordem certa e junta os gabaritos em record_chaos_log.
#   Parte sempre de entities, que não é alterado: rodar a célula de novo não aplica as falhas
#   duas vezes.
#
# FUNÇÃO DECLARADA:
# - inject_record_chaos(entities): devolve (landing_entities, gabarito). Ordem: atraso, valor fora
#   do padrão, coluna nova, duplicata e, por último, o gabarito do embaralhamento.
#
# FUNÇÕES NATIVAS:
# - dict(entities): cópia rasa do dicionário; trocar uma chave na cópia não altera o original.
# - sorted(conjunto): lista em ordem alfabética (conjuntos não têm ordem fixa).
# - functools.reduce(DataFrame.unionByName, lista): empilha todos os DataFrames da lista.
# - lista.append(x): acrescenta um item no fim da lista.
from functools import reduce


def inject_record_chaos(entities: dict[str, DataFrame]) -> tuple[dict[str, DataFrame], DataFrame]:
    """Aplica as falhas por registro. Devolve as entidades da landing e o gabarito dessas falhas."""
    landing = dict(entities)
    logs = []

    for entity in sorted(CHAOS_CONFIG["late_arrival"]["entities"]):
        landing[entity], log = inject_late_arrival(landing[entity], entity)
        logs.append(log)

    bad_entity = CHAOS_CONFIG["bad_value"]["entity"]
    landing[bad_entity], log = inject_bad_value(landing[bad_entity])
    logs.append(log)

    new_column_entity = CHAOS_CONFIG["new_column"]["entity"]
    landing[new_column_entity], log = inject_new_column(landing[new_column_entity])
    logs.append(log)

    for entity in sorted(CHAOS_CONFIG["duplicate_record"]["entities"]):
        landing[entity], log = inject_duplicates(landing[entity], entity)
        logs.append(log)

    if CHAOS_CONFIG["enabled"]:
        for entity in sorted(CHAOS_CONFIG["out_of_order"]["entities"]):
            logs.append(out_of_order_log(landing[entity], entity))

    return landing, reduce(DataFrame.unionByName, logs)


landing_entities, record_chaos_log = inject_record_chaos(entities)

In [0]:
# O QUE FAZ: confere as falhas sem gravar nada:
#   1. gabarito por tipo e entidade: total e até 2026-10-31 (período da Fase 1);
#   2. linhas na landing = linhas reais + duplicatas, por entidade (o atraso não cria nem remove
#      linhas, só muda o dia);
#   3. os itens com vírgula em 2026-10-07 e o primeiro dia com payment_gateway preenchida.
#
# FUNÇÕES NATIVAS:
# - F.sum(F.when(cond, 1).otherwise(0)): conta as linhas da condição dentro de um groupBy.
# - df.groupBy(...).agg(...): agrega por grupo.
# - Column.contains(texto): verdadeiro se o texto contiver o trecho.
display(
    record_chaos_log
    .groupBy("chaos_type", "entity")
    .agg(
        F.count("*").alias("total"),
        F.count_if(F.col("sim_date") <= PHASE_1_END).alias("ate_31_10"),
    )
    .orderBy("chaos_type", "entity")
)

duplicates_by_entity = {
    row["entity"]: row["count"]
    for row in record_chaos_log.where(F.col("chaos_type") == "duplicate_record").groupBy("entity").count().collect()
}
conferencia = spark.createDataFrame(
    [
        (entity, entities[entity].count(), duplicates_by_entity.get(entity, 0), landing_entities[entity].count())
        for entity in sorted(DAILY_ENTITIES)
    ],
    "entidade string, linhas_reais long, duplicatas long, linhas_landing long",
).withColumn("ok", F.col("linhas_reais") + F.col("duplicatas") == F.col("linhas_landing"))
display(conferencia)

display(
    landing_entities["order_items"]
    .where(F.col("price").contains(","))
    .select("order_id", "order_item_id", "price", "sim_date")
    .orderBy("order_id")
)
display(
    landing_entities["order_payments"]
    .groupBy(F.col("payment_gateway").isNotNull().alias("tem_gateway"))
    .agg(F.min("sim_date").alias("primeiro_dia"), F.max("sim_date").alias("ultimo_dia"), F.count("*").alias("linhas"))
)

## Controle da execução

Define quais dias simulados esta execução gera e registra a execução como `RUNNING` em `olist.ops.simulator_runs` (desenho A1).

- **Sem `sim_date_override`:** continua do dia seguinte ao último `SUCCESS`; sem histórico, começa em `SIM_START`.
- **Com `sim_date_override`:** reprocessa a partir do dia informado, que pode ser qualquer dia já gerado ou o próximo dia, nunca um dia à frente (não pode haver lacunas).
- O último dia simulado (`SIM_END`) é o dia da última compra do dataset, calculado na seção "Mudanças cadastrais". Quando todos os dias já foram gerados, a execução termina sem fazer nada.

In [0]:
# O QUE FAZ: declara as funções que decidem QUAIS dias esta execução gera.
#
# FUNÇÕES DECLARADAS:
# - get_last_success_date(): consulta ops.simulator_runs e devolve o último dia gerado com
#   SUCCESS, ou None se o simulador nunca rodou (MAX de zero linhas = NULL).
# - compute_window(last_success, override, dias, sim_start, sim_end): função pura que devolve
#   (início, fim) da janela ou None se não há mais dias. Regras: o próximo dia é o seguinte ao
#   último sucesso; override só pode repetir um dia já gerado ou o próximo (sem pular dias);
#   o fim não passa de sim_end.
#
# FUNÇÕES NATIVAS:
# - timedelta(days=n): intervalo de n dias; data + timedelta = outra data.
# - spark.sql(texto): executa SQL e devolve um DataFrame.
# - df.first(): ação; devolve a primeira linha (Row). Row["coluna"] lê o valor da coluna.
# - min(a, b): o menor entre dois valores (função nativa do Python).
from datetime import timedelta


def get_last_success_date() -> date | None:
    """Último dia simulado gerado com sucesso; None se o simulador nunca rodou."""
    return spark.sql(
        f"SELECT MAX(sim_date_end) AS ultimo FROM {RUNS_TABLE} WHERE status = 'SUCCESS'"
    ).first()["ultimo"]


def compute_window(
    last_success: date | None,
    override: date | None,
    dias: int,
    sim_start: date,
    sim_end: date,
) -> tuple[date, date] | None:
    """Intervalo de dias [início, fim] desta execução; None se não há mais dias a gerar. Função pura."""
    next_date = sim_start if last_success is None else last_success + timedelta(days=1)

    if override is not None and override > next_date:
        raise ValueError(f"sim_date_override ({override}) pularia dias: o próximo dia a gerar é {next_date}")
    start = override or next_date

    if start > sim_end:
        return None
    end = min(start + timedelta(days=dias - 1), sim_end)
    return start, end

In [0]:
# O QUE FAZ: calcula a janela desta execução e encerra o notebook com sucesso se não houver mais
#   dias a gerar. SIM_END (dia da última compra) é calculado na seção "Mudanças cadastrais".
#
# FUNÇÕES NATIVAS:
# - dbutils.notebook.exit(texto): termina o notebook nesse ponto, com sucesso, devolvendo o texto
#   (no job, a task termina como bem-sucedida).
# - a, b = tupla: desempacota a tupla (início, fim) em duas variáveis.
window = compute_window(get_last_success_date(), SIM_DATE_OVERRIDE, DIAS_POR_EXECUCAO, SIM_START, SIM_END)

if window is None:
    dbutils.notebook.exit(f"Nada a fazer: todos os dias até {SIM_END} já foram gerados.")

SIM_DATE_START, SIM_DATE_END = window
print(f"Dias desta execução: {SIM_DATE_START} a {SIM_DATE_END} (último dia da simulação: {SIM_END})")

In [0]:
# O QUE FAZ: registra o início da execução em ops.simulator_runs com status RUNNING.
#   Se o run_id já existe (retentativa do job), a linha é reaberta em vez de duplicada.
#
# RECURSOS SQL:
# - MERGE INTO alvo USING origem ON condição: compara a origem com a tabela alvo;
#   WHEN MATCHED THEN UPDATE  -> a linha já existe: atualiza;
#   WHEN NOT MATCHED THEN INSERT -> não existe: insere.
# - (SELECT :run_id AS run_id, ...): origem de uma linha só, montada com os valores da execução.
# - :run_id, :inicio, :fim: parameter markers. O valor vai em args={...}, com o tipo preservado
#   (date vira DATE) e sem risco de SQL injection. Eles substituem valores, não nomes de tabela:
#   por isso o nome da tabela vem da constante RUNS_TABLE na f-string.
# - current_timestamp(): data/hora atual.
# MERGE pelo run_id: se o job repetir a task com o mesmo run_id, a linha é reaberta em vez de duplicada
spark.sql(
    f"""
    MERGE INTO {RUNS_TABLE} AS t
    USING (SELECT :run_id AS run_id, :inicio AS sim_date_start, :fim AS sim_date_end) AS s
    ON t.run_id = s.run_id
    WHEN MATCHED THEN UPDATE SET
        sim_date_start = s.sim_date_start,
        sim_date_end   = s.sim_date_end,
        status         = 'RUNNING',
        started_at     = current_timestamp(),
        finished_at    = NULL,
        files_written  = NULL,
        rows_written   = NULL,
        error_message  = NULL
    WHEN NOT MATCHED THEN INSERT (run_id, sim_date_start, sim_date_end, status, started_at)
        VALUES (s.run_id, s.sim_date_start, s.sim_date_end, 'RUNNING', current_timestamp())
    """,
    args={"run_id": RUN_ID, "inicio": SIM_DATE_START, "fim": SIM_DATE_END},
)

In [0]:
# O QUE FAZ: mostra a linha desta execução em simulator_runs (deve estar RUNNING).
# - spark.sql(texto, args={...}): SQL com parameter marker; display() mostra o resultado.
display(spark.sql(f"SELECT * FROM {RUNS_TABLE} WHERE run_id = :run_id", args={"run_id": RUN_ID}))

## Escrita determinística

Grava um arquivo JSON Lines por entidade e por dia simulado, no caminho definido no desenho (A3):

```
raw_data/<entidade>/sim_date=AAAA-MM-DD/<entidade>_AAAA-MM-DD_part-000.json
```

O Spark não permite escolher o nome do arquivo de saída (gera `part-00000-<uuid>.json`). Por isso a escrita acontece em duas fases:

1. **Preparação:** cada entidade é gravada em `raw_data/_staging/<run_id>/<entidade>/`, particionada por `sim_date`, ordenada pela chave e com uma partição por dia (um arquivo por dia).
2. **Publicação:** só depois que todas as entidades foram preparadas, os dias da janela são limpos no destino e cada arquivo é movido para o nome definitivo. A pasta de preparação é apagada ao final, com ou sem erro.

Na publicação são aplicadas as falhas por arquivo (desenho A5): a linha de JSON inválido, inserida antes de mover o arquivo, e o reenvio, uma cópia publicada com o nome `..._part-000_retry-1.json`. As duas entram no gabarito.

In [0]:
# O QUE FAZ: constantes da escrita: ordem das linhas de cada entidade (garante arquivo idêntico
#   a cada geração), entidades de carga inicial e a pasta de preparação desta execução.
#
# RECURSOS:
# - set {...}: conjunto sem ordem e sem repetição; "x in conjunto" é uma busca rápida.
# - RUN_STAGING_DIR: uma pasta por run_id, para duas execuções nunca misturarem arquivos.
# Ordem das linhas dentro de cada arquivo: garante conteúdo idêntico a cada geração (desenho A3)
ENTITY_SORT_KEYS = {
    "orders": ["order_id"],
    "order_status_events": ["order_id", "event_sequence"],
    "order_items": ["order_id", "order_item_id"],
    "order_payments": ["order_id", "payment_sequential"],
    "order_reviews": ["review_id", "order_id"],
    "customers": ["customer_id", "change_sequence"],
    "sellers": ["seller_id", "change_sequence"],
    "products": ["product_id"],
    # geolocation não tem chave e tem linhas repetidas: ordena por todas as colunas
    "geolocation": [
        "geolocation_zip_code_prefix",
        "geolocation_lat",
        "geolocation_lng",
        "geolocation_city",
        "geolocation_state",
    ],
    "category_translation": ["product_category_name"],
}

# Entidades emitidas só no primeiro dia: fora dessa janela não precisam ser lidas.
# sellers não está aqui: com o CDC, também recebe UPDATE e DELETE em outros dias.
INITIAL_LOAD_ENTITIES = {"products", "geolocation", "category_translation"}

RUN_STAGING_DIR = f"{STAGING_DIR}/{RUN_ID}"

In [0]:
# O QUE FAZ: declara funções puras de caminho e de datas usadas na escrita.
#
# FUNÇÕES DECLARADAS:
# - day_dir(entity, day): pasta definitiva do dia: raw_data/<entidade>/sim_date=AAAA-MM-DD.
# - final_file_name(entity, day): nome determinístico: <entidade>_<dia>_part-000.json.
# - days_between(start, end): lista com todos os dias de start até end (incluindo os dois).
#
# FUNÇÕES NATIVAS:
# - range(n): números de 0 a n-1. (end - start).days: número de dias entre as datas.
# - [... for i in range(...)]: list comprehension, monta a lista em uma linha.
def day_dir(entity: str, day: date | str) -> str:
    """Pasta definitiva de um dia: raw_data/<entidade>/sim_date=AAAA-MM-DD"""
    return f"{RAW_DIR}/{entity}/sim_date={day}"


def final_file_name(entity: str, day: date | str) -> str:
    """Nome definitivo do arquivo: só entidade, dia e número da parte (sem uuid, sem horário)."""
    return f"{entity}_{day}_part-000.json"


def days_between(start: date, end: date) -> list[date]:
    """Todos os dias do intervalo fechado [start, end]."""
    return [start + timedelta(days=i) for i in range((end - start).days + 1)]

In [0]:
# O QUE FAZ: declara as funções das falhas que atuam no arquivo inteiro, aplicadas durante a
#   publicação (célula seguinte):
#   - corrupt_file: arquivo com uma linha de JSON inválido;
#   - file_resend: cópia do arquivo publicada com outro nome, como se a origem enviasse de novo.
#
# FUNÇÕES DECLARADAS:
# - retry_file_name(entity, day): nome da cópia reenviada: <entidade>_<dia>_part-000_retry-1.json.
#   O nome é diferente do original de propósito: para o Auto Loader é um arquivo novo, e a
#   duplicidade de negócio chega à bronze (desenho A5 e B6). E é fixo: reprocessar o dia gera o
#   mesmo nome, e o Auto Loader não lê de novo (A3).
# - corrupt_file(file): insere no meio do arquivo uma cópia cortada ao meio de uma das linhas, que
#   é JSON inválido, sempre na mesma posição. Devolve o número da linha inserida.
#
# FUNÇÕES NATIVAS:
# - dbutils.fs.head(caminho, max_bytes): lê o conteúdo do arquivo como texto, até max_bytes.
# - dbutils.fs.put(caminho, texto, overwrite=True): grava o texto no arquivo, substituindo-o.
#   Leitura e gravação pela mesma API (dbutils.fs) usada no resto da escrita (lição do reset).
# - str.splitlines(keepends=True): divide em linhas mantendo o "\n" no fim de cada uma.
# - lista.insert(posição, item): insere o item na posição; os seguintes andam uma casa.
# - texto[:n]: os n primeiros caracteres; "//" é a divisão inteira.
# - "".join(lista): junta as linhas de volta em um texto só.
MAX_CORRUPT_FILE_BYTES = 10 * 1024 * 1024  # 10 MB: bem acima de um arquivo diário (até ~1 MB)


def retry_file_name(entity: str, day: date | str) -> str:
    """Nome da cópia reenviada pela origem: diferente do original e fixo para o mesmo dia."""
    return f"{entity}_{day}_part-000_retry-1.json"


def corrupt_file(file) -> int:
    """Insere uma linha de JSON inválido no meio do arquivo. Devolve o número da linha inserida (1 = primeira)."""
    if file.size > MAX_CORRUPT_FILE_BYTES:
        raise RuntimeError(f"{file.path}: arquivo grande demais para a falha corrupt_file ({file.size} bytes)")
    lines = dbutils.fs.head(file.path, MAX_CORRUPT_FILE_BYTES).splitlines(keepends=True)
    middle = len(lines) // 2
    lines.insert(middle, lines[middle][: len(lines[middle]) // 2] + "\n")
    dbutils.fs.put(file.path, "".join(lines), overwrite=True)
    return middle + 1

In [0]:
# O QUE FAZ: declara a escrita em duas fases: prepara tudo na pasta _staging e, só se tudo deu
#   certo, limpa os dias no destino e publica os arquivos com o nome definitivo.
#
# FUNÇÕES DECLARADAS:
# - sort_columns(entity): ordem das linhas no arquivo. Normalmente, a chave; nas entidades com a
#   falha out_of_order, o hash da chave (ordem embaralhada, porém idêntica a cada geração) e, em
#   seguida, a chave, como desempate.
# - stage_entity(entity, df, start, end): grava os dias da janela em _staging/<run_id>/<entidade>/
#   em JSON, um arquivo por dia, com as linhas sempre na mesma ordem. Devolve o caminho. Na
#   entidade da falha new_column, os dias anteriores à data configurada são gravados SEM a coluna
#   nova (primeira gravação) e os demais com ela (segunda gravação, em modo append).
# - publish_entity(entity, staging_path): para cada pasta sim_date=... da preparação, confere que
#   há exatamente 1 part-*.json, aplica corrupt_file se for o arquivo configurado, move para o
#   caminho definitivo e, se o arquivo for sorteado para reenvio, publica a cópia retry-1.
#   Devolve (caminhos publicados, gabarito das falhas por arquivo).
# - write_window(entities, start, end): orquestra as 3 fases (preparar, limpar, publicar) e
#   devolve (arquivos, linhas, gabarito das falhas por arquivo). Linhas = todas as linhas dos
#   arquivos publicados, incluindo cópias e a linha inválida. Entidades de carga inicial só
#   entram se a janela incluir SIM_START.
#
# FUNÇÕES NATIVAS:
# - Column.between(a, b): verdadeiro se a <= valor <= b (inclui os dois extremos).
# - df.repartition("sim_date"): redistribui as linhas (shuffle) para que todas as linhas do mesmo
#   dia fiquem na mesma partição. Assim cada dia sai em um único arquivo.
# - df.sortWithinPartitions(colunas): ordena dentro de cada partição, sem ordenação global.
# - df.write.mode("overwrite"): sobrescreve a pasta de destino (aqui, só a preparação da execução);
#   mode("append") acrescenta arquivos sem apagar os que já existem.
# - enumerate(lista): devolve (posição, item); a posição 0 decide o modo overwrite.
# - .partitionBy("sim_date"): cria uma subpasta sim_date=<valor> por dia e tira a coluna de
#   dentro do arquivo (o valor fica só no nome da pasta).
# - .option("ignoreNullFields", False): grava "campo": null em vez de omitir o campo.
# - .json(caminho): grava em JSON Lines (um objeto JSON por linha).
# - dbutils.fs.ls(caminho): lista a pasta; cada item tem name, path, size e modificationTime.
#   Nomes de pasta terminam com "/".
# - str.startswith / str.endswith: o texto começa/termina com...; removeprefix tira o prefixo;
#   rstrip("/") tira a barra do fim.
# - dbutils.fs.mkdirs(caminho): cria a pasta (e as pastas acima, se faltarem).
# - dbutils.fs.mv(origem, destino): move/renomeia o arquivo.
# - dbutils.fs.rm(caminho, recurse=True): apaga a pasta e tudo dentro dela.
# - dbutils.fs.cp(origem, destino): copia o arquivo.
# - date.fromisoformat("AAAA-MM-DD"): texto -> date (o dia vem do nome da pasta, como texto).
# - lista.append(x) / lista += outra: acrescenta um item / os itens de outra lista.
# - spark.read.text(lista_de_caminhos).count(): lê os arquivos como texto (uma linha = uma linha
#   do DataFrame); em JSON Lines, linhas = registros.
# - raise RuntimeError(...): interrompe a execução com erro.
# - continue: pula para o próximo item do for.
def sort_columns(entity: str) -> list[Column]:
    """Ordem das linhas no arquivo: pela chave ou, nas entidades embaralhadas, pelo hash da chave."""
    keys = [F.col(k) for k in ENTITY_SORT_KEYS[entity]]
    if CHAOS_CONFIG["enabled"] and entity in CHAOS_CONFIG["out_of_order"]["entities"]:
        return [chaos_hash(ENTITY_SORT_KEYS[entity], "out_of_order"), *keys]
    return keys


def stage_entity(entity: str, df: DataFrame, start: date, end: date) -> str:
    """Grava os dias da janela na pasta de preparação: uma partição Spark por dia = um arquivo por dia."""
    path = f"{RUN_STAGING_DIR}/{entity}"
    window_df = df.where(F.col("sim_date").between(start, end))

    new_column = CHAOS_CONFIG["new_column"]
    if entity == new_column["entity"] and new_column["column"] in df.columns:
        # antes da data configurada, o arquivo não tem a coluna nova (falha new_column)
        parts = [
            window_df.where(F.col("sim_date") < new_column["from_date"]).drop(new_column["column"]),
            window_df.where(F.col("sim_date") >= new_column["from_date"]),
        ]
    else:
        parts = [window_df]

    for i, part in enumerate(parts):
        (
            part
            .repartition("sim_date")
            .sortWithinPartitions(*sort_columns(entity))
            .write
            .mode("overwrite" if i == 0 else "append")
            .partitionBy("sim_date")
            .option("ignoreNullFields", False)  # grava "campo": null em vez de omitir o campo
            .json(path)
        )
    return path


def publish_entity(entity: str, staging_path: str) -> tuple[list[str], list[tuple[date, str, str, str]]]:
    """Publica cada dia com o nome definitivo, aplicando as falhas por arquivo. Devolve (caminhos, gabarito)."""
    corrupt, resend = CHAOS_CONFIG["corrupt_file"], CHAOS_CONFIG["file_resend"]
    published, chaos_rows = [], []
    for day_folder in dbutils.fs.ls(staging_path):
        if not day_folder.name.startswith("sim_date="):
            continue  # _SUCCESS e arquivos de controle do Spark
        day = day_folder.name.removeprefix("sim_date=").rstrip("/")

        parts = [f for f in dbutils.fs.ls(day_folder.path) if f.name.startswith("part-") and f.name.endswith(".json")]
        if len(parts) != 1:
            raise RuntimeError(f"{entity} {day}: esperado 1 arquivo na preparação, encontrados {len(parts)}")

        if CHAOS_CONFIG["enabled"] and entity == corrupt["entity"] and day == str(corrupt["date"]):
            line = corrupt_file(parts[0])
            chaos_rows.append((date.fromisoformat(day), "corrupt_file", entity, f"{final_file_name(entity, day)}: linha {line} com JSON inválido"))

        final_path = f"{day_dir(entity, day)}/{final_file_name(entity, day)}"
        dbutils.fs.mkdirs(day_dir(entity, day))
        dbutils.fs.mv(parts[0].path, final_path)
        published.append(final_path)

        if entity in resend["entities"] and file_selected(entity, day, "file_resend", resend["pct"]):
            retry_path = f"{day_dir(entity, day)}/{retry_file_name(entity, day)}"
            dbutils.fs.cp(final_path, retry_path)
            published.append(retry_path)
            chaos_rows.append((date.fromisoformat(day), "file_resend", entity, retry_file_name(entity, day)))
    return published, chaos_rows


def write_window(entities: dict[str, DataFrame], start: date, end: date) -> tuple[int, int, list[tuple[date, str, str, str]]]:
    """Prepara todas as entidades, limpa os dias da janela no destino e publica. Devolve (arquivos, linhas, gabarito)."""
    includes_initial_load = start <= SIM_START <= end
    selected = {
        entity: df
        for entity, df in entities.items()
        if entity not in INITIAL_LOAD_ENTITIES or includes_initial_load
    }

    # 1. Preparação: se falhar aqui, o destino continua intacto
    staged = {entity: stage_entity(entity, df, start, end) for entity, df in selected.items()}

    # 2. Limpeza do destino: reprocessar um dia substitui o que existia, sem sobras
    for entity in staged:
        for day in days_between(start, end):
            dbutils.fs.rm(day_dir(entity, day), recurse=True)

    # 3. Publicação, com as falhas por arquivo
    published, file_chaos_rows = [], []
    for entity, path in staged.items():
        paths, chaos_rows = publish_entity(entity, path)
        published += paths
        file_chaos_rows += chaos_rows

    # Linhas de todos os arquivos publicados (JSON Lines: 1 linha = 1 registro, exceto a inválida)
    rows_written = spark.read.text(published).count() if published else 0
    return len(published), rows_written, file_chaos_rows

## Execução e finalização

Executa a escrita da janela e fecha o registro da execução em `olist.ops.simulator_runs`:

- **Sucesso:** `SUCCESS`, com a quantidade de arquivos e de linhas publicados. Só a partir daqui o progresso do simulador avança. O gabarito dos dias da janela é gravado antes, na mesma tentativa: arquivos e gabarito sempre correspondem.
- **Erro:** `FAILED`, com a mensagem de erro; o erro é relançado para que a execução (e a task do job) termine como falha.
- Em qualquer caso, a pasta de preparação da execução é apagada.

In [0]:
# O QUE FAZ: declara a função que fecha a linha desta execução em simulator_runs.
#
# FUNÇÃO DECLARADA:
# - finish_run(status, files, rows, error_message): UPDATE da linha do RUN_ID com o status final
#   (SUCCESS ou FAILED), o horário de término, os totais e a mensagem de erro. Parâmetros com
#   padrão None: na falha, só status e mensagem são informados.
#
# RECURSOS:
# - UPDATE tabela SET coluna = valor WHERE condição: altera as linhas que atendem à condição.
# - Parameter markers: None do Python vira NULL no SQL.
# - "int | None": anotação de tipo; o valor pode ser inteiro ou None.
def finish_run(status: str, files: int | None = None, rows: int | None = None, error_message: str | None = None) -> None:
    """Fecha a linha da execução atual em simulator_runs."""
    spark.sql(
        f"""
        UPDATE {RUNS_TABLE}
        SET status        = :status,
            finished_at   = current_timestamp(),
            files_written = :files,
            rows_written  = :rows,
            error_message = :error_message
        WHERE run_id = :run_id
        """,
        args={"status": status, "files": files, "rows": rows, "error_message": error_message, "run_id": RUN_ID},
    )

In [0]:
# O QUE FAZ: executa a escrita da janela (landing_entities, com as falhas) e grava o gabarito
#   dos mesmos dias, e então registra o resultado.
#   Erro -> FAILED com a mensagem e o erro é relançado (a task do job falha).
#   Sempre -> a pasta de preparação desta execução é apagada.
#   Sem erro -> SUCCESS com os totais (fora do try: um erro aqui não pode marcar FAILED uma
#   execução que já publicou todos os arquivos).
#   Gabarito = falhas por registro (record_chaos_log) + falhas por arquivo (devolvidas pela escrita).
#
# RECURSOS:
# - try / except Exception as error / finally: try executa; except roda só se houver erro;
#   finally roda sempre, com ou sem erro.
# - type(error).__name__: nome da classe do erro (ex.: "RuntimeError").
# - texto[:2000]: os primeiros 2.000 caracteres (limita o tamanho gravado na tabela).
# - raise (sozinho): relança o mesmo erro, com o traceback original.
try:
    files_written, rows_written, file_chaos_rows = write_window(landing_entities, SIM_DATE_START, SIM_DATE_END)
    chaos_log = record_chaos_log.unionByName(chaos_log_from_files(file_chaos_rows))
    chaos_rows = write_chaos_log(chaos_log, SIM_DATE_START, SIM_DATE_END)
except Exception as error:
    finish_run("FAILED", error_message=f"{type(error).__name__}: {error}"[:2000])
    raise
finally:
    dbutils.fs.rm(RUN_STAGING_DIR, recurse=True)

finish_run("SUCCESS", files=files_written, rows=rows_written)

print(f"SUCCESS: {files_written} arquivos e {rows_written} linhas publicados ({SIM_DATE_START} a {SIM_DATE_END})")
print(f"Gabarito: {chaos_rows} falhas registradas nesses dias")

In [0]:
# O QUE FAZ: validação: lista o arquivo publicado de cada entidade no primeiro dia da janela,
#   com o tamanho, e confirma que a pasta de preparação ficou vazia.
#
# RECURSOS:
# - try/except por entidade: dbutils.fs.ls lança erro se a pasta não existe (entidade sem
#   registro no dia), e a validação continua para as demais.
# - f.size: tamanho do arquivo em bytes.
# - "lista or 'vazia'": lista vazia conta como falso, então mostra 'vazia'.
for entity in ENTITY_SORT_KEYS:
    try:
        arquivos = [f"{f.name} ({f.size} bytes)" for f in dbutils.fs.ls(day_dir(entity, SIM_DATE_START))]
    except Exception:
        arquivos = ["(nenhum registro neste dia)"]
    print(f"{entity:22} {', '.join(arquivos)}")

print(f"\nPreparação: {[f.name for f in dbutils.fs.ls(STAGING_DIR)] or 'vazia'}")

In [0]:
# O QUE FAZ: mostra as 10 execuções mais recentes do simulador (status, janela, totais, erros).
# - ORDER BY started_at DESC LIMIT 10: ordena da mais nova para a mais antiga e pega 10.
display(spark.sql(f"SELECT * FROM {RUNS_TABLE} ORDER BY started_at DESC LIMIT 10"))